# Dashboard Dataset Preparation

## Objective

The purpose of this notebook is to prepare a clean analytical dataset for Power BI.

Instead of loading multiple raw tables into the dashboard, the data is transformed and combined into a single dataset using pandas. This approach simplifies the data model, improves dashboard performance, and follows common BI development practices.

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "06_Datasets" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "06_Datasets" / "processed"

In [4]:
orders = pd.read_csv(RAW_DATA / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DATA / "olist_products_dataset.csv")
customers = pd.read_csv(RAW_DATA / "olist_customers_dataset.csv")
reviews = pd.read_csv(RAW_DATA / "olist_order_reviews_dataset.csv")
categories = pd.read_csv(RAW_DATA / "product_category_name_translation.csv")

In [5]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "customers": customers,
    "reviews": reviews,
    "categories": categories,
}

pd.DataFrame(
    {
        "Rows": [df.shape[0] for df in tables.values()],
        "Columns": [df.shape[1] for df in tables.values()],
    },
    index=tables.keys(),
)

,Rows,Columns
orders,99441,8
order_items,112650,7
products,32951,9
customers,99441,5
reviews,99224,7
categories,71,2


## Merge Tables

In [18]:
dashboard = []
dashboard = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

dashboard = dashboard.merge(
    customers,
    on="customer_id",
    how="left"
)

dashboard = dashboard.merge(
    products,
    on="product_id",
    how="left"
)

dashboard = dashboard.merge(
    categories,
    on="product_category_name",
    how="left"
)

reviews = reviews.drop_duplicates(subset="order_id")

dashboard = dashboard.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

In [19]:
dashboard["purchase_date"] = pd.to_datetime(
    dashboard["order_purchase_timestamp"]
)

dashboard["delivered_date"] = pd.to_datetime(
    dashboard["order_delivered_customer_date"]
)

dashboard["delivery_days"] = (
    dashboard["delivered_date"] -
    dashboard["purchase_date"]
).dt.days

dashboard["purchase_year"] = (
    dashboard["purchase_date"].dt.year
)

dashboard["purchase_month"] = (
    dashboard["purchase_date"].dt.to_period("M").astype(str)
)

dashboard["total_item_value"] = (
    dashboard["price"] +
    dashboard["freight_value"]
)

In [20]:
dashboard = dashboard[
    [
        "order_id",
        "purchase_date",
        "order_status",
        "customer_unique_id",
        "customer_state",
        "seller_id",
        "product_category_name_english",
        "price",
        "freight_value",
        "delivery_days",
        "review_score",
    ]
]

## Validate Dataset

In [21]:
dashboard.info()
dashboard.describe(include="all")
dashboard.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 11 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   purchase_date                  112650 non-null  datetime64[ns]
 2   order_status                   112650 non-null  object        
 3   customer_unique_id             112650 non-null  object        
 4   customer_state                 112650 non-null  object        
 5   seller_id                      112650 non-null  object        
 6   product_category_name_english  111023 non-null  object        
 7   price                          112650 non-null  float64       
 8   freight_value                  112650 non-null  float64       
 9   delivery_days                  110196 non-null  float64       
 10  review_score                   111708 non-null  float64       
dtype

order_id                            0
purchase_date                       0
order_status                        0
customer_unique_id                  0
customer_state                      0
seller_id                           0
product_category_name_english    1627
price                               0
freight_value                       0
delivery_days                    2454
review_score                      942
dtype: int64

In [22]:
dashboard["product_category_name_english"] = (
    dashboard["product_category_name_english"]
    .fillna("Unknown")
)

In [23]:
dashboard["order_status"].value_counts()

order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

## Save Processed Dataset

In [48]:
output_file = PROCESSED_DATA / "dashboard_dataset.csv"

dashboard.to_csv(output_file, index=False)
